# 📐 Cuaderno 01 — Interpolación lineal y cuadrática

**Métodos Numéricos · Unidad 4: Ajuste de curvas e interpolación**

Primer acercamiento: unir dos puntos con una recta y tres puntos con una parábola, medir el error y decidir cuándo conviene subir de grado.

> 📖 Teoría: [Interpolación lineal y cuadrática](../docs/metodos/01_interpolacion_lineal_cuadratica.md)

In [ ]:
#@title ⚙️ Entorno (ejecuta PRIMERO)
import os, sys, importlib

try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False


def _buscar(nombre, saltos=6):
    """Busca un archivo junto al cuaderno, en la carpeta actual y en lib/ de los padres."""
    bases = [os.getcwd()]
    ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code
    if ruta_nb:
        bases.append(os.path.dirname(os.path.abspath(ruta_nb)))
    for base in list(bases):
        b = base
        for _ in range(saltos):
            for cand in (os.path.join(b, nombre), os.path.join(b, "lib", nombre)):
                if os.path.exists(cand):
                    return os.path.abspath(cand)
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


rutas = {n: _buscar(n) for n in ("robomat.py", "ajuste_curvas.py")}
faltan = [n for n, r in rutas.items() if r is None]
if faltan:
    raise FileNotFoundError(
        "No encontré " + ", ".join(faltan) +
        ". En Colab, sube la carpeta lib/ junto al cuaderno.")

_carpeta_lib = os.path.dirname(rutas["robomat.py"])
if _carpeta_lib not in sys.path:
    sys.path.insert(0, _carpeta_lib)

import robomat as _robomat
importlib.reload(_robomat)
from robomat import *               # noqa: F401,F403

import ajuste_curvas as ac
importlib.reload(ac)

import numpy as np
import matplotlib.pyplot as plt

try:
    from ipywidgets import interact, FloatSlider
except ImportError:
    interact = None

print("✅ Entorno cargado desde:", _carpeta_lib)
print("✅ Entorno:", "Google Colab" if ES_COLAB else "local")

### 🎯 Objetivos

- Construir el polinomio de **primer grado** que pasa por dos nodos, ecuación (18.2).
- Construir el polinomio **cuadrático** que pasa por tres nodos, ecuaciones (18.3)–(18.6).
- Medir el error con la ecuación (18.18) y compararlo con el error verdadero.
- Explicar por qué el intervalo y la posición de los nodos cambian el error.

**Referencia del libro:** capítulo 5, secciones 18.1.1 y 18.1.2, ejemplos 18.1, 18.2 y 18.4.

## 1. El problema: estimar $\ln 2$ con pocos datos

Se conocen algunos valores de $\ln x$ y se quiere estimar $\ln 2=0.6931472$.
Empezamos con los nodos más separados y luego los acercamos:

| nodo | $x$ | $f(x)$ |
|---|---|---|
| 0 | 1 | 0 |
| 1 | 4 | 1.386294 |
| 2 | 6 | 1.791759 |
| 3 | 5 | 1.609438 |

El punto 3 no se usa al principio: sirve para **estimar el error**.

In [ ]:
x = np.array([1.0, 4.0, 6.0, 5.0])
f = np.array([0.0, 1.386294, 1.791759, 1.609438])
VERDADERO = 0.6931472

xs = np.linspace(0.8, 6.4, 300)
plt.figure(figsize=(8, 5))
plt.plot(xs, np.log(xs), label="ln x (verdadera)", lw=2)
plt.plot(x[:3], f[:3], "o", ms=8, label="nodos")
plt.axvline(2.0, color="0.7", ls=":")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Datos disponibles para estimar ln 2")
plt.show()

## 2. Interpolación lineal

Por triángulos semejantes, la recta que pasa por $(x_0,f_0)$ y $(x_1,f_1)$ es

$$f_1(x)=f(x_0)+\frac{f(x_1)-f(x_0)}{x_1-x_0}(x-x_0). \tag{18.2}$$

Probamos dos intervalos: $[1,6]$ y $[1,4]$. El más corto debe dar mejor
resultado porque la recta se parece más a la curva.

In [ ]:
def lineal(x0, f0, x1, f1, xp):
    return f0 + (f1 - f0) / (x1 - x0) * (xp - x0)

pares = [(1.0, 0.0, 6.0, 1.791759), (1.0, 0.0, 4.0, 1.386294)]
for (x0, f0, x1, f1) in pares:
    aprox = lineal(x0, f0, x1, f1, 2.0)
    error = abs(aprox - VERDADERO) / VERDADERO * 100
    print(f"intervalo [{x0}, {x1}] -> f(2) = {aprox:.7f}   error = {error:.1f}%")

**Lectura.** El libro reporta $0.3583519$ (error $48.3\%$) y $0.4620981$
(error $33.3\%$). La diferencia entre ambos resultados es enorme y no viene de
la aritmética: viene de **dónde están los nodos**.

## 3. Interpolación cuadrática

Con tres nodos se usa la forma anidada

$$f_2(x)=b_0+b_1(x-x_0)+b_2(x-x_0)(x-x_1), \tag{18.3}$$

donde

$$b_0=f(x_0),\qquad b_1=\frac{f(x_1)-f(x_0)}{x_1-x_0},\qquad
b_2=\frac{\dfrac{f(x_2)-f(x_1)}{x_2-x_1}-\dfrac{f(x_1)-f(x_0)}{x_1-x_0}}{x_2-x_0}. \tag{18.4–18.6}$$

Es la misma tabla de diferencias divididas que veremos en el cuaderno 02.

In [ ]:
x3 = x[:3]
f3 = f[:3]
b = ac.coeficientes_newton(x3, f3)
print("b0, b1, b2 =", b)
f2_2 = ac.evaluar_newton(x3, b, 2.0)
print(f"f2(2) = {f2_2:.7f}   error = {abs(f2_2 - VERDADERO)/VERDADERO*100:.1f}%")

# Comprobación: pasa exactamente por los tres nodos
print("en los nodos:", np.allclose(ac.evaluar_newton(x3, b, x3), f3))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(xs, np.log(xs), label="ln x", lw=2)
plt.plot(xs, [lineal(1.0, 0.0, 4.0, 1.386294, t) for t in xs],
         "--", label="lineal [1,4]")
plt.plot(xs, ac.evaluar_newton(x3, b, xs), label="cuadrática [1,4,6]")
plt.plot(x3, f3, "o", ms=8)
plt.axvline(2.0, color="0.7", ls=":")
plt.ylim(-0.5, 2.0); plt.xlabel("x"); plt.ylabel("f(x)")
plt.legend(); plt.grid(alpha=0.3); plt.title("Lineal vs cuadrática")
plt.show()

## 4. Error de la interpolación

El error de truncamiento para grado $n$ es

$$R_n=\frac{f^{(n+1)}(\xi)}{(n+1)!}\prod_{i=0}^{n}(x-x_i), \tag{18.16}$$

y si no se conoce $f$ se estima con **un punto extra**:

$$R_n\cong f[x_{n+1},x_n,\dots,x_0]\prod_{i=0}^{n}(x-x_i). \tag{18.18}$$

Para la cuadrática usamos el punto extra $(5, 1.609438)$.

In [ ]:
b4 = ac.coeficientes_newton(x, f)
R2 = b4[3] * (2.0 - x[0]) * (2.0 - x[1]) * (2.0 - x[2])
print("b3 =", b4[3])
print(f"error estimado R2 = {R2:.7f}")
print(f"error verdadero   = {abs(f2_2 - VERDADERO):.7f}")
print("¿mismo orden de magnitud?", np.isclose(R2, abs(f2_2 - VERDADERO), rtol=0.6))

## 5. Laboratorio: mover el nodo intermedio

La calidad de la interpolación depende de dónde están los nodos. Mueve $x_1$
y observa el error de la recta que une $x_0=1$ con $x_1$ para estimar $\ln 2$.

In [ ]:
if interact is not None:
    def error_lineal(x1=4.0):
        aprox = lineal(1.0, 0.0, x1, np.log(x1), 2.0)
        print(f"x1 = {x1:.2f} -> f(2) = {aprox:.6f}, error = {abs(aprox-VERDADERO)/VERDADERO*100:.2f}%")

    interact(error_lineal, x1=FloatSlider(min=2.05, max=6.0, step=0.05, value=4.0))
else:
    print("ipywidgets no está instalado; prueba manualmente con error_lineal(x1=...).")

## 6. Autoevaluación

**Pregunta 1.** Con los nodos $(1,0)$ y $(6,1.791759)$, ¿cuánto vale la
interpolación lineal en $x=2$? Escribe tu respuesta numérica.

In [ ]:
respuesta_1 = 0.0     # <-- cambia este valor

correcta_1 = lineal(1.0, 0.0, 6.0, 1.791759, 2.0)
print("✅ Correcto" if verificar(respuesta_1, correcta_1, tol=1e-4)
      else f"❌ Revisa: usa la ecuación (18.2). Valor esperado: {correcta_1:.7f}")

**Pregunta 2.** ¿Cuál es el coeficiente $b_2$ de la cuadrática que pasa por
$(1,0)$, $(4,1.386294)$ y $(6,1.791759)$?

In [ ]:
respuesta_2 = 0.0     # <-- cambia este valor

correcta_2 = ac.coeficientes_newton([1.0, 4.0, 6.0], [0.0, 1.386294, 1.791759])[2]
print("✅ Correcto" if verificar(respuesta_2, correcta_2, tol=1e-3)
      else f"❌ Revisa la ecuación (18.6). Valor esperado: {correcta_2:.7f}")

**Pregunta 3 (conceptual).** Si el punto a estimar está **fuera** del intervalo
de los nodos, ¿qué pasa con el error? Responde en la celda siguiente y
justifica con la ecuación (18.16).

In [ ]:
# Escribe tu respuesta como texto y ejecútala.
respuesta_3 = '''
Escribe aquí tu explicación.
'''
print(respuesta_3)

## 7. Resumen

- Dos puntos → recta (18.2); tres puntos → parábola (18.3)–(18.6).
- El polinomio de grado $n$ que pasa por $n+1$ nodos es **único**.
- El error depende del producto $\prod(x-x_i)$: nodos cercanos y a ambos lados
  del punto son mejores.
- Un punto extra permite **estimar** el error sin conocer la función (18.18).

En el cuaderno 02 generalizamos estas ideas a cualquier número de nodos.